# 文字位置のアノテーションを BIO タグ列に変換する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/10_sequence_labeling/spans_to_bio.ipynb)

NLP クックブック 10-01 ／ 系列ラベリング

「何文字目から何文字目が人名」という形式 (doccano や brat の出力) のアノテーションを、トークンごとの BIO タグに変換したい。

In [ ]:
!pip install -q fugashi unidic-lite

In [ ]:
from fugashi import Tagger

tagger = Tagger()

def tokenize_with_offsets(text):
    pos, out = 0, []
    for w in tagger(text):
        start = text.index(w.surface, pos)
        pos = start + len(w.surface)
        out.append((w.surface, start, pos))
    return out

def spans_to_bio(text, spans):
    """spans: [(開始文字位置, 終了文字位置, ラベル), ...]"""
    tokens = tokenize_with_offsets(text)
    tags = ["O"] * len(tokens)
    for s, e, label in spans:
        inside = [i for i, (_, ts, te) in enumerate(tokens) if ts < e and te > s]   # スパンと重なるトークン
        for n, i in enumerate(inside):
            tags[i] = ("B-" if n == 0 else "I-") + label
        if inside and (tokens[inside[0]][1] != s or tokens[inside[-1]][2] != e):
            print(f"  警告: 境界がトークンと合わない: {text[s:e]!r} → {''.join(tokens[i][0] for i in inside)!r}")
    return [t for t, _, _ in tokens], tags

# doccano の JSONL 形式: {"text": ..., "label": [[start, end, label], ...]}
record = {"text": "山田太郎は2020年に京都大学を卒業し、東京都で働いている。",
          "label": [[0, 4, "PER"], [5, 10, "DATE"], [11, 15, "ORG"], [20, 22, "LOC"]]}
tokens, tags = spans_to_bio(record["text"], record["label"])
for tok, tag in zip(tokens, tags):
    print(f"{tok}\t{tag}")

- 「東京」だけがアノテーションされているのに、トークナイザが「東京都」を 1 語にする、といった境界のずれが必ず起きる。件数を数えて、多ければ文字単位のタグ付けやサブワードの利用を検討する。
- I- で始まる系列を許す IOB1、末尾と単独を区別する BIOES (BILOU) などの変種がある。データセットがどれかを確認する。